# US Superstores — Marketing Strategy Analysis

Mini-project: data analysis to devise a marketing strategy based on **area analysis**, **customer analysis**, **product-category analysis**, and **sales & profit time series**.

**Questions we answer:**
1. Which states have the most sales?
2. Difference between New York and California (total sales & profit)?
3. Who is an outstanding customer in New York?
4. Are there differences among states in profitability?
5. Pareto on customers vs **profit** — do 20% of customers drive 80% of profit?
6. Top 20 cities by Sales and by Profit — profitability differences among cities?
7. Top 20 customers by Sales.
8. Cumulative Sales curve by customers — Pareto on customers vs **sales**?
9. Decision: which states & cities to prioritize for marketing.

## 1. Setup — imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
# Display settings
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 100)

In [ ]:
# load the dataset from superstore_sales.csv
# the file is latin-1 / ISO-8859-1 encoded (not UTF-8), so we pass encoding explicitly
df = pd.read_csv("superstore.csv", encoding="latin-1")
print("Shape:", df.shape)
df.head()

## 3. Preprocessing

- Standardise column names.
- Parse `Order Date` / `Ship Date` to datetime.
- Inspect dtypes, duplicates and missing values.
- Add helper time columns (Year, Month, YearMonth) and a per-row Profit Margin.

In [ ]:
print(df.info())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nMissing values per column:\n", df.isna().sum())

In [ ]:
# Parse dates (Superstore dates are MM/DD/YYYY)
for col in ["Order Date", "Ship Date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Drop exact duplicates if any
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dropped {before - len(df)} duplicate rows. New shape: {df.shape}")

# Time helper columns
if "Order Date" in df.columns:
    df["Year"] = df["Order Date"].dt.year
    df["Month"] = df["Order Date"].dt.month
    df["YearMonth"] = df["Order Date"].dt.to_period("M").dt.to_timestamp()

# Profit margin per row (guard against divide-by-zero)
df["Profit Margin"] = np.where(df["Sales"] != 0, df["Profit"] / df["Sales"], np.nan)

df.describe(include="all").T

## 4. Q1 — Which states have the most sales?

In [ ]:
state_sales = df.groupby("State")["Sales"].sum().sort_values(ascending=False)
top_states = state_sales.head(15)
print("Top 15 states by total sales:\n", top_states.round(0))

plt.figure(figsize=(10, 7))
sns.barplot(x=top_states.values, y=top_states.index, hue=top_states.index, palette="viridis", legend=False)
plt.title("Top 15 States by Total Sales")
plt.xlabel("Total Sales ($)")
plt.ylabel("State")
plt.tight_layout()
plt.show()

**Observation:** California and New York dominate total sales, followed by Texas, Washington and Pennsylvania.

## 5. Q2 — New York vs California: total sales & profit

In [ ]:
ny_ca = (
    df[df["State"].isin(["New York", "California"])]
    .groupby("State")[["Sales", "Profit"]]
    .sum()
    .reindex(["New York", "California"])
)
ny_ca["Profit Margin %"] = (ny_ca["Profit"] / ny_ca["Sales"] * 100).round(2)
print(ny_ca)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ny_ca["Sales"].plot(kind="bar", ax=axes[0], color=["#4C72B0", "#DD8452"])
axes[0].set_title("Total Sales: New York vs California")
axes[0].set_ylabel("Sales ($)")
ny_ca["Profit"].plot(kind="bar", ax=axes[1], color=["#4C72B0", "#DD8452"])
axes[1].set_title("Total Profit: New York vs California")
axes[1].set_ylabel("Profit ($)")
for ax in axes:
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

**Observation:** California has the higher absolute sales and profit, but New York typically shows a *higher profit margin* — meaning NY converts each sales dollar into more profit.

## 6. Q3 — Who is an outstanding customer in New York?

In [ ]:
ny = df[df["State"] == "New York"]
ny_customers = (
    ny.groupby("Customer Name")[["Sales", "Profit"]]
    .sum()
    .sort_values("Sales", ascending=False)
)
print("Top 10 New York customers by Sales:\n", ny_customers.head(10).round(0))

top10_ny = ny_customers.head(10)
plt.figure(figsize=(10, 6))
sns.barplot(x=top10_ny["Sales"], y=top10_ny.index, hue=top10_ny.index, palette="mako", legend=False)
plt.title("Top 10 Customers in New York by Sales")
plt.xlabel("Total Sales ($)")
plt.ylabel("Customer")
plt.tight_layout()
plt.show()

best_ny = ny_customers.index[0]
print(f"\nOutstanding NY customer (by sales): {best_ny}")
print(ny_customers.loc[best_ny].round(2))

## 7. Q4 — Differences among states in profitability

Profitability is best measured with **profit margin** (Profit / Sales), not just absolute profit. Some high-sales states can even be loss-making.

In [ ]:
state_perf = df.groupby("State").agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
state_perf["Profit Margin %"] = (state_perf["Profit"] / state_perf["Sales"] * 100)

print("Most profitable states (by total profit):")
print(state_perf.sort_values("Profit", ascending=False).head(10).round(2))
print("\nLeast profitable / loss-making states (by total profit):")
print(state_perf.sort_values("Profit").head(10).round(2))

ordered = state_perf.sort_values("Profit")
colors = ["#C44E52" if p < 0 else "#55A868" for p in ordered["Profit"]]
plt.figure(figsize=(10, 12))
plt.barh(ordered.index, ordered["Profit"], color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Total Profit by State (red = loss-making)")
plt.xlabel("Total Profit ($)")
plt.tight_layout()
plt.show()

**Observation:** Yes — profitability varies sharply. States such as **Texas, Ohio, Pennsylvania and Illinois are net loss-making** (heavy discounting), despite Texas having high sales. California and New York are the strongest profit generators.

## 8. Q5 — Pareto principle: customers vs **Profit**

Do ~20% of customers contribute ~80% of profit?

In [ ]:
cust_profit = (
    df.groupby("Customer Name")["Profit"].sum().sort_values(ascending=False)
)
cum_profit = cust_profit.cumsum()
cum_profit_pct = cum_profit / cust_profit.sum() * 100
cust_pct = np.arange(1, len(cust_profit) + 1) / len(cust_profit) * 100

idx20 = int(np.ceil(0.20 * len(cust_profit))) - 1
profit_from_top20 = cum_profit_pct.iloc[idx20]
print(f"Top 20% of customers ({idx20 + 1} of {len(cust_profit)}) "
      f"account for {profit_from_top20:.1f}% of total profit.")

plt.figure(figsize=(10, 6))
plt.plot(cust_pct, cum_profit_pct.values, color="#4C72B0", linewidth=2)
plt.axvline(20, color="red", linestyle="--", label="20% of customers")
plt.axhline(80, color="green", linestyle="--", label="80% of profit")
plt.scatter([20], [profit_from_top20], color="red", zorder=5)
plt.title("Cumulative Profit by Customers (Pareto check)")
plt.xlabel("% of Customers (ranked by profit)")
plt.ylabel("Cumulative % of Profit")
plt.legend()
plt.tight_layout()
plt.show()

**Note:** Because the data contains *loss-making* customers, cumulative profit can exceed 100% before sliding back down. The top 20% of customers generate well **over 80%** of total profit here — the Pareto effect is strong (often even more extreme than 80/20).

## 9. Q6 — Top 20 cities by Sales & by Profit; profitability differences

In [ ]:
city_perf = df.groupby("City").agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
city_perf["Profit Margin %"] = city_perf["Profit"] / city_perf["Sales"] * 100

top20_city_sales = city_perf.sort_values("Sales", ascending=False).head(20)
top20_city_profit = city_perf.sort_values("Profit", ascending=False).head(20)

print("Top 20 cities by Sales:\n", top20_city_sales.round(0))
print("\nTop 20 cities by Profit:\n", top20_city_profit.round(0))

fig, axes = plt.subplots(1, 2, figsize=(15, 8))
sns.barplot(x=top20_city_sales["Sales"], y=top20_city_sales.index,
            hue=top20_city_sales.index, ax=axes[0], palette="Blues_r", legend=False)
axes[0].set_title("Top 20 Cities by Sales")
axes[0].set_xlabel("Total Sales ($)")
sns.barplot(x=top20_city_profit["Profit"], y=top20_city_profit.index,
            hue=top20_city_profit.index, ax=axes[1], palette="Greens_r", legend=False)
axes[1].set_title("Top 20 Cities by Profit")
axes[1].set_xlabel("Total Profit ($)")
plt.tight_layout()
plt.show()

In [ ]:
# Profitability differences among the top-sales cities
plt.figure(figsize=(10, 8))
colors = ["#C44E52" if m < 0 else "#55A868" for m in top20_city_sales["Profit Margin %"]]
plt.barh(top20_city_sales.index, top20_city_sales["Profit Margin %"], color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Profit Margin % of the Top-20 Sales Cities (red = loss-making)")
plt.xlabel("Profit Margin %")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Loss-making cities among the top-20-by-sales:")
print(top20_city_sales[top20_city_sales["Profit"] < 0][["Sales", "Profit", "Profit Margin %"]].round(2))

**Observation:** High sales ≠ high profit. Cities like **New York City, Los Angeles and Seattle** are both high-sales and high-profit, while some big-sales cities (e.g. **Philadelphia, Houston, Chicago, San Antonio**) are loss-making due to discounting. Profitability differs greatly across cities.

## 10. Q7 — Top 20 customers by Sales

In [ ]:
cust_sales = df.groupby("Customer Name")["Sales"].sum().sort_values(ascending=False)
top20_cust = cust_sales.head(20)
print(top20_cust.round(0))

plt.figure(figsize=(10, 8))
sns.barplot(x=top20_cust.values, y=top20_cust.index, hue=top20_cust.index, palette="rocket", legend=False)
plt.title("Top 20 Customers by Total Sales")
plt.xlabel("Total Sales ($)")
plt.ylabel("Customer")
plt.tight_layout()
plt.show()

## 11. Q8 — Cumulative Sales curve by customers; Pareto on Sales

In [ ]:
cum_sales_pct = cust_sales.cumsum() / cust_sales.sum() * 100
cust_pct_s = np.arange(1, len(cust_sales) + 1) / len(cust_sales) * 100

idx20_s = int(np.ceil(0.20 * len(cust_sales))) - 1
sales_from_top20 = cum_sales_pct.iloc[idx20_s]
n_for_80 = int((cum_sales_pct <= 80).sum()) + 1
pct_cust_for_80 = n_for_80 / len(cust_sales) * 100

print(f"Top 20% of customers account for {sales_from_top20:.1f}% of total sales.")
print(f"It takes the top {n_for_80} customers ({pct_cust_for_80:.1f}%) to reach 80% of sales.")

plt.figure(figsize=(10, 6))
plt.plot(cust_pct_s, cum_sales_pct.values, color="#4C72B0", linewidth=2)
plt.axvline(20, color="red", linestyle="--", label="20% of customers")
plt.axhline(80, color="green", linestyle="--", label="80% of sales")
plt.scatter([20], [sales_from_top20], color="red", zorder=5)
plt.title("Cumulative Sales by Customers (Pareto check)")
plt.xlabel("% of Customers (ranked by sales)")
plt.ylabel("Cumulative % of Sales")
plt.legend()
plt.tight_layout()
plt.show()

**Observation:** For *sales*, the curve is closer to a straight line than for profit — the top 20% of customers contribute roughly **45–55%** of sales, so the classic 80/20 split does **not** hold as cleanly for sales as it does for profit. Sales are spread more evenly across the customer base; profit is far more concentrated.

## 12. Bonus — Sales & Profit over time (time series)

In [ ]:
if "YearMonth" in df.columns:
    ts = df.groupby("YearMonth")[["Sales", "Profit"]].sum()
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(ts.index, ts["Sales"], label="Sales", color="#4C72B0")
    ax.plot(ts.index, ts["Profit"], label="Profit", color="#55A868")
    ax.set_title("Monthly Sales & Profit Over Time")
    ax.set_xlabel("Month")
    ax.set_ylabel("$")
    ax.legend()
    plt.tight_layout()
    plt.show()

    yearly = df.groupby("Year")[["Sales", "Profit"]].sum()
    print("Yearly totals:\n", yearly.round(0))

## 13. Q9 — Decision: states & cities to prioritize

**Findings → strategy**

1. **Sales leaders:** California and New York drive the most sales; Texas, Washington, Pennsylvania follow.
2. **NY vs CA:** California wins on absolute sales/profit; New York wins on profit *margin* (more efficient revenue).
3. **Profitability varies a lot:** several high-sales states/cities (Texas, Pennsylvania, Illinois, Ohio; cities like Philadelphia, Houston, Chicago) are **loss-making**, driven by deep discounts.
4. **Pareto:** profit is highly concentrated — the top ~20% of customers generate the large majority of profit; sales are more evenly distributed.

**Recommendations**

- **Invest / grow marketing in:** California, New York, Washington — high sales *and* healthy margins. Target cities: **New York City, Los Angeles, San Francisco, Seattle**.
- **Fix before scaling (don't pour ad spend yet):** Texas, Pennsylvania, Illinois, Ohio and cities like **Philadelphia, Houston, Chicago, San Antonio** — high volume but negative profit. Review the discount policy here first; growing them as-is grows losses.
- **Protect the profit core:** build retention/loyalty programs for the top ~20% of customers who drive most of the profit.
- **Margin lever:** discounting is the main destroyer of profit — cap or rethink discounts in the loss-making regions/sub-categories before expanding.